In [0]:
%pip install xgboost lightgbm catboost
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 96.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 142.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.7/96.7 MB 85.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.4/206.4 MB 83.9 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 192.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [catboost]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import joblib
import pandas as pd
import mlflow
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

In [0]:
# Exact raw input columns coming from your Silver table
RAW_INPUT_COLS = [
    "amount_usd", "hours_since_last_txn", "txn_count_last_24h", "distance_from_home_km",
    "card_age_months", "customer_age", "account_balance_usd", "cvv_retry_count",
    "velocity_score", "time_of_day_hour", "day_of_week", "merchant_risk_score",
    "prior_disputes", "merchant_category", "card_type", "auth_method", "channel",
    "device_type", "is_foreign_transaction", "is_new_merchant", "used_vpn",
    "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"
]

# Exact column order produced by preprocess(train) excluding ['id', 'is_fraud']
TRAIN_FEATURE_ORDER = [
    "amount_usd", "hours_since_last_txn", "txn_count_last_24h", "distance_from_home_km",
    "card_age_months", "customer_age", "account_balance_usd", "cvv_retry_count",
    "velocity_score", "time_of_day_hour", "day_of_week", "merchant_risk_score",
    "prior_disputes", "merchant_category", "card_type", "auth_method", "channel",
    "device_type", "is_foreign_transaction", "is_new_merchant", "used_vpn",
    "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt",
    "amount_percentage", "balance_remaining", "travel_speed_kmh", "is_late_night",
    "total_risk_flags"
]

BOOL_COLS = [
    "is_foreign_transaction", "is_new_merchant", "used_vpn",
    "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"
]

CAT_COLS = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]

In [0]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Financial ratios
    df["amount_percentage"] = df["amount_usd"] / (df["account_balance_usd"] + 1e-5)
    df["balance_remaining"] = df["account_balance_usd"] - df["amount_usd"]

    # 2. Behavioral & impossible-travel features
    df["travel_speed_kmh"] = df["distance_from_home_km"] / (df["hours_since_last_txn"] + 0.1)
    df["is_late_night"] = df["time_of_day_hour"].between(0, 5).astype(int)

    # 3. Convert booleans to int & sum risk flags (fillna(False) protects against streaming nulls)
    df[BOOL_COLS] = df[BOOL_COLS].fillna(False).astype(int)
    df["total_risk_flags"] = df[BOOL_COLS].sum(axis=1)

    return df[TRAIN_FEATURE_ORDER]

In [0]:
class FraudEnsembleModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.xgb = joblib.load(context.artifacts["xgb_path"])
        self.lgb = joblib.load(context.artifacts["lgb_path"])
        self.cat = joblib.load(context.artifacts["cat_path"])

    def predict(self, context, model_input: pd.DataFrame) -> pd.Series:
        df = preprocess(model_input)

        # XGBoost & LightGBM require 'category' dtype
        df_tree = df.copy()
        for col in CAT_COLS:
            df_tree[col] = df_tree[col].astype("category")

        # CatBoost requires string/int for cat_features
        df_cat = df.copy()
        for col in CAT_COLS:
            df_cat[col] = df_cat[col].astype(str)

        p_xgb = self.xgb.predict_proba(df_tree)[:, 1]
        p_lgb = self.lgb.predict_proba(df_tree)[:, 1]
        p_cat = self.cat.predict_proba(df_cat)[:, 1]

        fraud_prob = 0.35 * p_xgb + 0.35 * p_lgb + 0.30 * p_cat

        return pd.Series(fraud_prob)

In [0]:
artifacts = {
    "xgb_path": "/Volumes/fraud_detection/data_ingection/models_vol/model_xgbc.pkl",
    "lgb_path": "/Volumes/fraud_detection/data_ingection/models_vol/model_lgbmc.pkl",
    "cat_path": "/Volumes/fraud_detection/data_ingection/models_vol/model_catbc.pkl",
}

# Grab 5 rows from Silver to infer input/output schema and test predict() automatically
sample_input = spark.table("fraud_dectection.data_ingection.silver").limit(5).toPandas()[RAW_INPUT_COLS]

with mlflow.start_run(run_name="fraud_ensemble_v1"):
    mlflow.pyfunc.log_model(
        artifact_path="fraud_ensemble",
        python_model=FraudEnsembleModel(),
        artifacts=artifacts,
        pip_requirements=["xgboost", "lightgbm", "catboost", "scikit-learn", "pandas", "joblib"],
        input_example=sample_input,
        registered_model_name="fraud_dectection.data_ingection.fraud_ensemble_model"
    )

2026/11/26 13:35:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-6b9d6d6b-3548.cloud.databricks.com/ml/experiments/3120520413065737/models/m-0855160ffb57462eb9a4ea1b9242da03?o=7474648555546597
2026/11/26 13:35:27 INFO mlflow.pyfunc: Inferring model signature from input example
/databricks/python/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at in

/databricks/python/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Registered model 'fraud_dectection.data_ingection.fraud_ensemble_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/14 [00:00<?, ?it/s]

🔗 Created version '1' of model 'fraud_dectection.data_ingection.fraud_ensemble_model': https://dbc-6b9d6d6b-3548.cloud.databricks.com/explore/data/models/fraud_dectection/data_ingection/fraud_ensemble_model/version/1?o=7474648555546597
